# ConvNeXt-Base Hyperparameter Optimization

Notebook 12 confirmed ConvNeXt-Base as the best architecture for SGFood-233 across all learning rates and metrics. This notebook pushes its performance ceiling by optimizing:

1. **Weight decay** — regularization strength
2. **Data augmentation** — training-time image transforms
3. **Label smoothing** — soft target regularization

Each phase uses the best values from previous phases (sequential dependency). After tuning, we train a final model and compare against all baselines.

In [ ]:
# Imports
import os
import sys
import json
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import timm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import CSVLogger
from sklearn.metrics import f1_score, confusion_matrix
from tqdm.auto import tqdm

from utils import (
    Config, set_seed, FoodClassifier, LocalImageDataset,
    get_train_transforms, get_val_transforms, create_dataloaders
)
from hyperparameter_tuning import (
    TunableFoodClassifier, ExperimentConfig,
    get_augmentation_transforms
)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"PyTorch {torch.__version__} | Device: {device}")
if device == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load dataset splits and create DataLoaders
DATASET_ROOT = '../foodsg-233_train_val_test_split'
NUM_WORKERS = 4
BATCH_SIZE = 64
NUM_CLASSES = 233
RESULTS_DIR = 'hp_results/convnext_tuning'
os.makedirs(RESULTS_DIR, exist_ok=True)

val_transform = get_val_transforms()

val_dataset = LocalImageDataset(DATASET_ROOT, split='val', transform=val_transform)
test_dataset = LocalImageDataset(DATASET_ROOT, split='test', transform=val_transform)

val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0
)
test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0
)

def evaluate_comprehensive(model, dataloader, device, num_classes=233, k=5):
    """Evaluate model with comprehensive metrics."""
    model.eval().to(device)
    all_preds, all_labels, all_top5 = [], [], []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc='Evaluating'):
            images = images.to(device)
            logits = model(images)
            preds = logits.argmax(dim=1)
            top5 = logits.topk(k, dim=1).indices
            top5_correct = (top5 == labels.to(device).unsqueeze(1)).any(dim=1)
            all_preds.append(preds.cpu())
            all_labels.append(labels)
            all_top5.append(top5_correct.cpu())

    preds = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    top5_correct = torch.cat(all_top5).numpy()

    return {
        'top1_acc': (preds == labels).mean(),
        'top5_acc': top5_correct.mean(),
        'macro_f1': f1_score(labels, preds, average='macro'),
        'weighted_f1': f1_score(labels, preds, average='weighted'),
        'preds': preds,
        'labels': labels,
        'per_class_acc': {
            c: (preds[labels == c] == c).mean()
            for c in range(num_classes) if (labels == c).sum() > 0
        },
    }

print(f"Val:  {len(val_dataset):,} images")
print(f"Test: {len(test_dataset):,} images")

In [ ]:
# Load best LR from notebook 12
lr_sweep_path = 'hp_results/lr_sweep/lr_sweep_results.json'

if os.path.exists(lr_sweep_path):
    with open(lr_sweep_path, 'r') as f:
        lr_sweep_results = json.load(f)
    best_lr_entry = max(lr_sweep_results['ConvNeXt-Base'], key=lambda r: r['best_val_acc'])
    BEST_LR = best_lr_entry['lr']
    print(f"Best LR from notebook 12: {BEST_LR:.0e} (val_acc={best_lr_entry['best_val_acc']:.4f})")
else:
    BEST_LR = 1e-4
    print(f"LR sweep results not found, using default: {BEST_LR:.0e}")

# Starting config
print(f"\nStarting configuration:")
print(f"  Model:     convnext_base")
print(f"  LR:        {BEST_LR:.0e}")
print(f"  WD:        1e-4 (to be tuned)")
print(f"  Augment:   basic (to be tuned)")
print(f"  Smoothing: 0.0 (to be tuned)")
print(f"  Scheduler: cosine")
print(f"  Optimizer: AdamW")

## Section B: HP Sweeps

Sequential HP sweeps: each phase uses best values from previous phases. 5 epochs per config to keep compute manageable.

In [ ]:
# Phase 1: Weight Decay Sweep
WEIGHT_DECAYS = [0, 1e-5, 1e-4, 1e-3, 1e-2]
SWEEP_EPOCHS = 5

wd_results_path = os.path.join(RESULTS_DIR, 'wd_sweep_results.json')

if os.path.exists(wd_results_path):
    with open(wd_results_path, 'r') as f:
        wd_results = json.load(f)
    print(f"Loaded cached WD results from {wd_results_path}")
else:
    wd_results = []

    for wd in WEIGHT_DECAYS:
        print(f"\n{'='*50}")
        print(f"Weight Decay = {wd}")
        print(f"{'='*50}")

        set_seed(42)

        config = ExperimentConfig(
            model_name='convnext_base',
            num_classes=NUM_CLASSES,
            learning_rate=BEST_LR,
            weight_decay=wd,
            batch_size=BATCH_SIZE,
            max_epochs=SWEEP_EPOCHS,
            scheduler='cosine',
            augmentation='basic',
            label_smoothing=0.0,
            optimizer='AdamW',
        )

        train_transform = get_augmentation_transforms('basic')
        train_dataset = LocalImageDataset(DATASET_ROOT, split='train', transform=train_transform)
        train_loader = torch.utils.data.DataLoader(
            train_dataset, batch_size=BATCH_SIZE, shuffle=True,
            num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0
        )

        model = TunableFoodClassifier(config)

        checkpoint_cb = ModelCheckpoint(
            monitor='val_acc', mode='max', save_top_k=1,
            filename=f'convnext_wd{wd}' + '-{epoch:02d}-{val_acc:.4f}',
            dirpath=os.path.join(RESULTS_DIR, 'checkpoints'),
        )

        trainer = pl.Trainer(
            max_epochs=SWEEP_EPOCHS,
            accelerator='gpu' if device == 'cuda' else 'cpu',
            devices=1,
            callbacks=[checkpoint_cb],
            enable_progress_bar=True,
            log_every_n_steps=50,
        )

        trainer.fit(model, train_loader, val_loader)

        best_val_acc = checkpoint_cb.best_model_score.item() if checkpoint_cb.best_model_score else 0.0
        val_top5 = trainer.callback_metrics.get('val_top5_acc', torch.tensor(0.0)).item()

        wd_results.append({
            'weight_decay': wd,
            'best_val_acc': best_val_acc,
            'val_top5_acc': val_top5,
        })

        print(f"  Best val_acc: {best_val_acc:.4f}")

        del model, trainer
        torch.cuda.empty_cache()

    with open(wd_results_path, 'w') as f:
        json.dump(wd_results, f, indent=2)

# Find best WD
best_wd_entry = max(wd_results, key=lambda r: r['best_val_acc'])
BEST_WD = best_wd_entry['weight_decay']
print(f"\nBest Weight Decay: {BEST_WD} (val_acc={best_wd_entry['best_val_acc']:.4f})")

In [ ]:
# Phase 2: Augmentation Sweep (using best LR + best WD)
AUGMENTATIONS = ['none', 'basic', 'moderate', 'strong', 'autoaugment']

aug_results_path = os.path.join(RESULTS_DIR, 'aug_sweep_results.json')

if os.path.exists(aug_results_path):
    with open(aug_results_path, 'r') as f:
        aug_results = json.load(f)
    print(f"Loaded cached augmentation results from {aug_results_path}")
else:
    aug_results = []

    for aug in AUGMENTATIONS:
        print(f"\n{'='*50}")
        print(f"Augmentation = {aug} | LR={BEST_LR:.0e} | WD={BEST_WD}")
        print(f"{'='*50}")

        set_seed(42)

        config = ExperimentConfig(
            model_name='convnext_base',
            num_classes=NUM_CLASSES,
            learning_rate=BEST_LR,
            weight_decay=BEST_WD,
            batch_size=BATCH_SIZE,
            max_epochs=SWEEP_EPOCHS,
            scheduler='cosine',
            augmentation=aug,
            label_smoothing=0.0,
            optimizer='AdamW',
        )

        train_transform = get_augmentation_transforms(aug)
        train_dataset = LocalImageDataset(DATASET_ROOT, split='train', transform=train_transform)
        train_loader = torch.utils.data.DataLoader(
            train_dataset, batch_size=BATCH_SIZE, shuffle=True,
            num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0
        )

        model = TunableFoodClassifier(config)

        checkpoint_cb = ModelCheckpoint(
            monitor='val_acc', mode='max', save_top_k=1,
            filename=f'convnext_aug{aug}' + '-{epoch:02d}-{val_acc:.4f}',
            dirpath=os.path.join(RESULTS_DIR, 'checkpoints'),
        )

        trainer = pl.Trainer(
            max_epochs=SWEEP_EPOCHS,
            accelerator='gpu' if device == 'cuda' else 'cpu',
            devices=1,
            callbacks=[checkpoint_cb],
            enable_progress_bar=True,
            log_every_n_steps=50,
        )

        trainer.fit(model, train_loader, val_loader)

        best_val_acc = checkpoint_cb.best_model_score.item() if checkpoint_cb.best_model_score else 0.0
        val_top5 = trainer.callback_metrics.get('val_top5_acc', torch.tensor(0.0)).item()

        aug_results.append({
            'augmentation': aug,
            'best_val_acc': best_val_acc,
            'val_top5_acc': val_top5,
        })

        print(f"  Best val_acc: {best_val_acc:.4f}")

        del model, trainer
        torch.cuda.empty_cache()

    with open(aug_results_path, 'w') as f:
        json.dump(aug_results, f, indent=2)

# Find best augmentation
best_aug_entry = max(aug_results, key=lambda r: r['best_val_acc'])
BEST_AUG = best_aug_entry['augmentation']
print(f"\nBest Augmentation: {BEST_AUG} (val_acc={best_aug_entry['best_val_acc']:.4f})")

In [ ]:
# Phase 3: Label Smoothing Sweep (using best LR + WD + augmentation)
LABEL_SMOOTHINGS = [0, 0.05, 0.1, 0.15, 0.2]

ls_results_path = os.path.join(RESULTS_DIR, 'ls_sweep_results.json')

if os.path.exists(ls_results_path):
    with open(ls_results_path, 'r') as f:
        ls_results = json.load(f)
    print(f"Loaded cached label smoothing results from {ls_results_path}")
else:
    ls_results = []

    for ls in LABEL_SMOOTHINGS:
        print(f"\n{'='*50}")
        print(f"Label Smoothing = {ls} | LR={BEST_LR:.0e} | WD={BEST_WD} | Aug={BEST_AUG}")
        print(f"{'='*50}")

        set_seed(42)

        config = ExperimentConfig(
            model_name='convnext_base',
            num_classes=NUM_CLASSES,
            learning_rate=BEST_LR,
            weight_decay=BEST_WD,
            batch_size=BATCH_SIZE,
            max_epochs=SWEEP_EPOCHS,
            scheduler='cosine',
            augmentation=BEST_AUG,
            label_smoothing=ls,
            optimizer='AdamW',
        )

        train_transform = get_augmentation_transforms(BEST_AUG)
        train_dataset = LocalImageDataset(DATASET_ROOT, split='train', transform=train_transform)
        train_loader = torch.utils.data.DataLoader(
            train_dataset, batch_size=BATCH_SIZE, shuffle=True,
            num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0
        )

        model = TunableFoodClassifier(config)

        checkpoint_cb = ModelCheckpoint(
            monitor='val_acc', mode='max', save_top_k=1,
            filename=f'convnext_ls{ls}' + '-{epoch:02d}-{val_acc:.4f}',
            dirpath=os.path.join(RESULTS_DIR, 'checkpoints'),
        )

        trainer = pl.Trainer(
            max_epochs=SWEEP_EPOCHS,
            accelerator='gpu' if device == 'cuda' else 'cpu',
            devices=1,
            callbacks=[checkpoint_cb],
            enable_progress_bar=True,
            log_every_n_steps=50,
        )

        trainer.fit(model, train_loader, val_loader)

        best_val_acc = checkpoint_cb.best_model_score.item() if checkpoint_cb.best_model_score else 0.0
        val_top5 = trainer.callback_metrics.get('val_top5_acc', torch.tensor(0.0)).item()

        ls_results.append({
            'label_smoothing': ls,
            'best_val_acc': best_val_acc,
            'val_top5_acc': val_top5,
        })

        print(f"  Best val_acc: {best_val_acc:.4f}")

        del model, trainer
        torch.cuda.empty_cache()

    with open(ls_results_path, 'w') as f:
        json.dump(ls_results, f, indent=2)

# Find best label smoothing
best_ls_entry = max(ls_results, key=lambda r: r['best_val_acc'])
BEST_LS = best_ls_entry['label_smoothing']
print(f"\nBest Label Smoothing: {BEST_LS} (val_acc={best_ls_entry['best_val_acc']:.4f})")

In [ ]:
# Tuning summary table and bar chart
BASELINE_ACC = 0.8455  # ConvNeXt cleaned baseline

tuning_phases = [
    ('Baseline (lr=1e-4, wd=1e-4, basic, ls=0)', BASELINE_ACC),
    (f'+ Best WD = {BEST_WD}', best_wd_entry['best_val_acc']),
    (f'+ Best Aug = {BEST_AUG}', best_aug_entry['best_val_acc']),
    (f'+ Best LS = {BEST_LS}', best_ls_entry['best_val_acc']),
]

tuning_df = pd.DataFrame(tuning_phases, columns=['Configuration', 'Val Accuracy'])
tuning_df['Improvement'] = tuning_df['Val Accuracy'].diff().fillna(0)
tuning_df['Cumulative Gain'] = tuning_df['Val Accuracy'] - BASELINE_ACC

print("Tuning Summary")
print("="*80)
display(tuning_df)

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Phase-by-phase accuracy
ax1 = axes[0]
phase_labels = ['Baseline', f'WD={BEST_WD}', f'Aug={BEST_AUG}', f'LS={BEST_LS}']
accs = [p[1] for p in tuning_phases]
bars = ax1.bar(phase_labels, accs, color=['#95a5a6', '#3498db', '#2ecc71', '#e74c3c'],
               edgecolor='black')
ax1.set_ylabel('Val Accuracy', fontsize=12)
ax1.set_title('Cumulative HP Optimization', fontsize=14)
ax1.axhline(y=BASELINE_ACC, color='gray', linestyle='--', alpha=0.5)
for bar, acc in zip(bars, accs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{acc:.4f}', ha='center', va='bottom', fontsize=10)
ax1.tick_params(axis='x', rotation=15)

# Individual sweep comparison
ax2 = axes[1]
all_sweep_data = [
    ('WD', [r['weight_decay'] for r in wd_results], [r['best_val_acc'] for r in wd_results]),
    ('Aug', [r['augmentation'] for r in aug_results], [r['best_val_acc'] for r in aug_results]),
    ('LS', [r['label_smoothing'] for r in ls_results], [r['best_val_acc'] for r in ls_results]),
]

# Show range for each sweep
for i, (name, vals, accs_list) in enumerate(all_sweep_data):
    ax2.barh(i, max(accs_list) - min(accs_list), left=min(accs_list),
             height=0.5, alpha=0.7, label=f'{name} range')
    ax2.scatter(max(accs_list), i, color='green', s=100, zorder=5, marker='*')

ax2.set_yticks(range(len(all_sweep_data)))
ax2.set_yticklabels([d[0] for d in all_sweep_data])
ax2.set_xlabel('Val Accuracy', fontsize=12)
ax2.set_title('HP Sensitivity Range', fontsize=14)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'tuning_summary.png'), dpi=150, bbox_inches='tight')
plt.show()

### Discussion: HP Impact

**Key findings** (fill in after running):

- **Weight decay**: Controls overfitting vs underfitting tradeoff. For fine-tuning pretrained ConvNeXt, moderate WD (1e-4 to 1e-3) typically works best — too high and the pretrained features are over-regularized.
- **Augmentation**: Food images are sensitive to extreme distortions — a heavily augmented curry may become unrecognizable. Moderate augmentation (RandAugment) usually helps, but AutoAugment (designed for ImageNet) may not be optimal for food-specific features.
- **Label smoothing**: With 233 classes, many pairs are visually similar (e.g., different noodle soups). Light smoothing (0.05–0.1) can help by preventing the model from being overconfident on ambiguous samples.

## Section C: Final Training

Train ConvNeXt-Base with the optimized configuration for a full run with early stopping.

In [ ]:
# Train final model with optimized config
FINAL_EPOCHS = 15
FINAL_PATIENCE = 5
CHECKPOINT_DIR = 'checkpoints/sgfood233_convnext_base_tuned'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("Final Training Configuration")
print("="*50)
print(f"  Model:          convnext_base")
print(f"  LR:             {BEST_LR:.0e}")
print(f"  Weight Decay:   {BEST_WD}")
print(f"  Augmentation:   {BEST_AUG}")
print(f"  Label Smooth:   {BEST_LS}")
print(f"  Max Epochs:     {FINAL_EPOCHS}")
print(f"  Early Stopping: patience={FINAL_PATIENCE}")
print(f"  Scheduler:      cosine")
print(f"  Optimizer:      AdamW")

set_seed(42)

final_config = ExperimentConfig(
    model_name='convnext_base',
    num_classes=NUM_CLASSES,
    learning_rate=BEST_LR,
    weight_decay=BEST_WD,
    batch_size=BATCH_SIZE,
    max_epochs=FINAL_EPOCHS,
    scheduler='cosine',
    augmentation=BEST_AUG,
    label_smoothing=BEST_LS,
    optimizer='AdamW',
)

train_transform = get_augmentation_transforms(BEST_AUG)
train_dataset = LocalImageDataset(DATASET_ROOT, split='train', transform=train_transform)
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0
)

final_model = TunableFoodClassifier(final_config)

checkpoint_cb = ModelCheckpoint(
    monitor='val_acc', mode='max', save_top_k=1,
    filename='epoch={epoch:02d}-val_acc={val_acc:.4f}',
    dirpath=CHECKPOINT_DIR,
)

early_stop_cb = EarlyStopping(
    monitor='val_acc', mode='max', patience=FINAL_PATIENCE, verbose=True
)

lr_monitor = LearningRateMonitor(logging_interval='epoch')

logger = CSVLogger(RESULTS_DIR, name='final_training')

trainer = pl.Trainer(
    max_epochs=FINAL_EPOCHS,
    accelerator='gpu' if device == 'cuda' else 'cpu',
    devices=1,
    callbacks=[checkpoint_cb, early_stop_cb, lr_monitor],
    logger=logger,
    enable_progress_bar=True,
    log_every_n_steps=50,
)

trainer.fit(final_model, train_loader, val_loader)

print(f"\nBest val_acc: {checkpoint_cb.best_model_score:.4f}")
print(f"Checkpoint: {checkpoint_cb.best_model_path}")

In [ ]:
# Training curves
metrics_file = os.path.join(logger.log_dir, 'metrics.csv')

if os.path.exists(metrics_file):
    metrics_df = pd.read_csv(metrics_file)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Loss curves
    ax1 = axes[0]
    if 'train_loss' in metrics_df.columns:
        train_loss = metrics_df.dropna(subset=['train_loss'])
        ax1.plot(train_loss['epoch'], train_loss['train_loss'], label='Train Loss', color='#e74c3c')
    if 'val_loss' in metrics_df.columns:
        val_loss = metrics_df.dropna(subset=['val_loss'])
        ax1.plot(val_loss['epoch'], val_loss['val_loss'], label='Val Loss', color='#3498db')
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('Training & Validation Loss', fontsize=14)
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Accuracy curves
    ax2 = axes[1]
    if 'train_acc' in metrics_df.columns:
        train_acc = metrics_df.dropna(subset=['train_acc'])
        ax2.plot(train_acc['epoch'], train_acc['train_acc'], label='Train Acc', color='#e74c3c')
    if 'val_acc' in metrics_df.columns:
        val_acc = metrics_df.dropna(subset=['val_acc'])
        ax2.plot(val_acc['epoch'], val_acc['val_acc'], label='Val Acc', color='#3498db')
    # Overlay baseline
    ax2.axhline(y=BASELINE_ACC, color='gray', linestyle='--', alpha=0.6, label=f'Baseline ({BASELINE_ACC:.4f})')
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy', fontsize=12)
    ax2.set_title('Training & Validation Accuracy', fontsize=14)
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Training metrics file not found.")

## Section D: Comprehensive Evaluation

Compare the tuned ConvNeXt against all baselines on the test set with comprehensive metrics.

In [ ]:
# Load all 4 models and evaluate on test set
ALL_MODELS = {
    'ResNet50 (baseline)': {
        'path': 'checkpoints/sgfood233_resnet50/epoch=09-val_acc=0.7509.ckpt',
        'backbone': 'resnet50',
        'loader': 'foodclassifier',
    },
    'ViT-Base (baseline)': {
        'path': 'checkpoints/sgfood233_vit_base_patch16_224/epoch=09-val_acc=0.8015.ckpt',
        'backbone': 'vit_base_patch16_224',
        'loader': 'foodclassifier',
    },
    'ConvNeXt-Base (baseline)': {
        'path': 'checkpoints/sgfood233_convnext_base_cleaned/epoch=09-val_acc=0.8455.ckpt',
        'backbone': 'convnext_base',
        'loader': 'foodclassifier',
    },
    'ConvNeXt-Base (tuned)': {
        'path': checkpoint_cb.best_model_path,
        'backbone': 'convnext_base',
        'loader': 'tunable',
        'config': final_config,
    },
}

final_results = {}

for name, info in ALL_MODELS.items():
    print(f"\n{'='*50}")
    print(f"Evaluating: {name}")
    print(f"{'='*50}")

    if info['loader'] == 'foodclassifier':
        model = FoodClassifier.load_from_checkpoint(
            info['path'],
            backbone_name=info['backbone'],
            num_classes=NUM_CLASSES,
        )
    else:
        model = TunableFoodClassifier.load_from_checkpoint(
            info['path'], config=info['config']
        )

    test_metrics = evaluate_comprehensive(model, test_loader, device)
    final_results[name] = test_metrics

    print(f"  Top-1: {test_metrics['top1_acc']:.4f} | Top-5: {test_metrics['top5_acc']:.4f} "
          f"| Macro-F1: {test_metrics['macro_f1']:.4f} | Weighted-F1: {test_metrics['weighted_f1']:.4f}")

    del model
    torch.cuda.empty_cache()

# Final 4x4 comparison table
print("\n\n" + "="*90)
print("FINAL COMPARISON: ALL MODELS ON TEST SET")
print("="*90)

final_rows = []
for name in ALL_MODELS:
    m = final_results[name]
    final_rows.append({
        'Model': name,
        'Top-1 Acc': f"{m['top1_acc']:.4f}",
        'Top-5 Acc': f"{m['top5_acc']:.4f}",
        'Macro-F1': f"{m['macro_f1']:.4f}",
        'Weighted-F1': f"{m['weighted_f1']:.4f}",
    })

final_df = pd.DataFrame(final_rows)
display(final_df)

# Per-class analysis: did tuning help the hardest classes?
baseline_pca = final_results['ConvNeXt-Base (baseline)']['per_class_acc']
tuned_pca = final_results['ConvNeXt-Base (tuned)']['per_class_acc']

# Find classes where tuning helped most
improvements = {}
for c in baseline_pca:
    if c in tuned_pca:
        improvements[c] = tuned_pca[c] - baseline_pca[c]

sorted_imp = sorted(improvements.items(), key=lambda x: x[1], reverse=True)

print("\nTop 10 classes with biggest improvement from tuning:")
for c, imp in sorted_imp[:10]:
    print(f"  Class {c}: {baseline_pca[c]:.3f} → {tuned_pca[c]:.3f} (+{imp:.3f})")

print("\nTop 10 classes with biggest regression from tuning:")
for c, imp in sorted_imp[-10:]:
    print(f"  Class {c}: {baseline_pca[c]:.3f} → {tuned_pca[c]:.3f} ({imp:+.3f})")

# Top confused pairs for tuned model
tuned_preds = final_results['ConvNeXt-Base (tuned)']['preds']
tuned_labels = final_results['ConvNeXt-Base (tuned)']['labels']

cm = confusion_matrix(tuned_labels, tuned_preds, labels=range(NUM_CLASSES))
np.fill_diagonal(cm, 0)  # Zero out correct predictions

# Find top confused pairs
confused_pairs = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if cm[i, j] > 0:
            confused_pairs.append((i, j, cm[i, j]))

confused_pairs.sort(key=lambda x: x[2], reverse=True)

print("\nTop 10 confused pairs (tuned model):")
for true_c, pred_c, count in confused_pairs[:10]:
    print(f"  True={true_c} predicted as {pred_c}: {count} errors")

# Save final results
final_summary = {
    name: {
        'top1_acc': float(m['top1_acc']),
        'top5_acc': float(m['top5_acc']),
        'macro_f1': float(m['macro_f1']),
        'weighted_f1': float(m['weighted_f1']),
    }
    for name, m in final_results.items()
}

with open(os.path.join(RESULTS_DIR, 'final_comparison.json'), 'w') as f:
    json.dump(final_summary, f, indent=2)

print(f"\nResults saved to {os.path.join(RESULTS_DIR, 'final_comparison.json')}")